# G3 rep2 — Integrated Analysis

Six panels covering the full training dynamics of `DiT_mini_parity_N4096_D36_G3_even_rep2`:

1. **Sample raster** — per-sample 4-state trajectory
2. **Transition matrix** — transition counts and probabilities over a key window
3. **Vector field / score landscape** — at ep 49, 14251, 119377, 492388 (σ=1.0)
4. **DSM loss σ∈[0.2,2.0] vs training step** — train/test/random
5. **DSM loss vs σ at key checkpoints** — log-log curves
6. **Attractor basin profiles** — three directions, three epochs

In [ ]:
import os, sys
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

PROJECT_ROOT = "/n/home12/binxuwang/Github/DiffusionAttnConsistency"
sys.path.insert(0, PROJECT_ROOT)

# Publication-quality defaults
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top']   = False
plt.rcParams['figure.dpi'] = 120

# ── Imports ───────────────────────────────────────────────────────
from core.vector_field_lib import (
    load_training_data, load_model, eval_field_on_grid,
    make_plane_hash, project_to_basis,
    plot_vector_field_2d, plot_denoiser_target_2d, DEFAULT_SAVEROOT,
)
from core.basin_lib import basin_plot_profiles
from scripts.plot_sample_evolution import (
    load_data, plot_raster, compute_transition_matrix,
    plot_transition_heatmap_both,
)
from scripts.plot_sigma_loss_evolution import (
    load_sigma_data, bin_mean, _draw_sigma_panel, SIGMA_BINS,
)

%matplotlib inline
print('Imports OK')

In [ ]:
# ── Config ────────────────────────────────────────────────────────
EXP_NAME  = "DiT_mini_parity_N4096_D36_G3_even_rep2"
SAVEROOT  = DEFAULT_SAVEROOT
EXP_DIR   = os.path.join(SAVEROOT, EXP_NAME)
FIGDIR    = os.path.join(PROJECT_ROOT, "figures", "G3rep2_analysis")
os.makedirs(FIGDIR, exist_ok=True)

# Vector field checkpoints (use cached σ=1.0)
VF_EPOCHS  = [49, 14251, 119377, 492388]
VF_SIGMA   = 1.0
VF_CACHE   = os.path.join(EXP_DIR, "vector_field_cache")
PLANE_HASH = "191059703a35"   # sha1 prefix of (x_a, x_b, x_c) anchors
RANGE_TAG  = "a-1.75_3.75"
NGRID      = 50

# Basin checkpoints
BASIN_EPOCHS = [7017, 20309, 492388]
BASIN_SIGMA  = 1.0
BASIN_N      = 30
BASIN_CACHE  = os.path.join(EXP_DIR, "basin_analysis", "line_cache")
BASIN_LABELS = {
    7017:   "ep 7017 (pre rule-learning)",
    20309:  "ep 20309 (post rule-learning)",
    492388: "ep 492388 (memorization onset)",
}
BASIN_COLORS = {7017: 'C0', 20309: 'C1', 492388: 'C2'}

# sigma_loss which_idx: [ep49, ep14251, ep119377, ep492388]
SIGMA_LOSS_IDX = [10, 26, 32, 36]

def savefig_both(fig, name):
    """Save PNG + PDF with fonttype 42."""
    for ext in ('.png', '.pdf'):
        path = os.path.join(FIGDIR, name + ext)
        fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f"  Saved: {name}.png/pdf")

print(f"FIGDIR: {FIGDIR}")

## Panel 1 — Sample Raster

In [ ]:
d = load_data(EXP_NAME, SAVEROOT)
fig = plot_raster(d, EXP_NAME, figdir=FIGDIR, save=False)
savefig_both(fig, '01_sample_raster')
plt.show()

## Panel 2 — Transition Heatmap

Show both unnormalized (count) and normalized (probability) heatmaps summed over a key window.
Two windows: early training (ep 1k–20k, rule-learning transition) and late (ep 20k–500k, memorization onset).

In [ ]:
T_count, T_prob, epochs_ev, ep_mid = compute_transition_matrix(d)

for win_label, win_lo, win_hi in [
    ('rule_learning',   1_000,   50_000),
    ('memorization', 50_000, 800_000),
]:
    mask = (ep_mid >= win_lo) & (ep_mid < win_hi)
    T_win = T_count[mask].sum(axis=0)   # (4,4) summed counts
    fig = plot_transition_heatmap_both(
        T_win,
        title=f"Transition counts — ep {win_lo//1000}k–{win_hi//1000}k"
    )
    savefig_both(fig, f'02_transition_heatmap_{win_label}')
    plt.show()

## Panel 3 — Vector Field / Score Landscape

Three-sample plane (x_a, x_b, x_c) with L2-scaled axes. Using cached evaluations (σ=1.0).

In [ ]:
# Reconstruct the plane geometry from training data
x_train = load_training_data(EXP_NAME, saveroot=SAVEROOT).numpy()  # (N,D)

# Anchors (must match what was used when generating the cache):
x_a = x_train[0].copy()
# x_b: valid novel — flip bits 0+1 within group-0 ((-1)²=+1, parity preserved)
x_b = x_a.copy(); x_b[0] *= -1; x_b[1] *= -1
# x_c: invalid — flip bit-0 only (breaks group-0 parity)
x_c = x_a.copy(); x_c[0] *= -1

# Verify plane hash matches cache
computed_hash = make_plane_hash(x_a, x_b, x_c)
print(f"Plane hash: {computed_hash}  (expect prefix: {PLANE_HASH})")
assert computed_hash.startswith(PLANE_HASH), f"Hash mismatch! {computed_hash}"

# Build L2-scaled basis
ab = x_b - x_a
ac = x_c - x_a
L_ab = float(np.linalg.norm(ab)); v_ab = ab / L_ab
ac_perp = ac - ac.dot(v_ab) * v_ab
L_perp  = float(np.linalg.norm(ac_perp)); v_ac = ac_perp / L_perp

margin = 1.75
alpha_ax = np.linspace(-margin, L_ab + margin, NGRID)
beta_ax  = np.linspace(-margin, L_perp + margin, NGRID)
A, B = np.meshgrid(alpha_ax, beta_ax, indexing='ij')  # (NGRID, NGRID)

# Marker positions
xc_alpha = float(ac.dot(v_ab))
xc_beta  = float(np.linalg.norm(ac_perp))

print(f"L_ab={L_ab:.2f}  L_perp={L_perp:.2f}")
print(f"x_a @ (0,0)  x_b @ ({L_ab:.2f},0)  x_c @ ({xc_alpha:.2f},{xc_beta:.2f})")


In [ ]:
def load_vf_cache(epoch, sigma=VF_SIGMA):
    fname = os.path.join(VF_CACHE,
        f"vf_ep{epoch:06d}_{PLANE_HASH}_{RANGE_TAG}_sig{sigma:.4f}_n{NGRID}.npz")
    return np.load(fname)

# Plot: 2 rows × 4 cols  (row0=denoiser target, row1=score magnitude + arrows)
fig, axes = plt.subplots(2, len(VF_EPOCHS), figsize=(16, 7),
                          gridspec_kw={'hspace': 0.15, 'wspace': 0.07})

markers = [
    (0,       0,       'x_a', 'white'),
    (L_ab,    0,       'x_b', 'cyan'),
    (xc_alpha, xc_beta, 'x_c', 'yellow'),
    (xd_alpha, xd_beta, 'x_d', 'orange'),
]

for ei, epoch in enumerate(VF_EPOCHS):
    res = load_vf_cache(epoch)
    D_pull = res['D_pull'].reshape(NGRID, NGRID, -1)  # (G,G,D)
    score  = res['score'].reshape(NGRID, NGRID, -1)

    # Project D_pull onto plane basis for arrows
    disp_u, disp_v = project_to_basis(
        res['D_pull'], v_ab, v_ac)   # (N,), (N,)
    disp_u = disp_u.reshape(NGRID, NGRID)
    disp_v = disp_v.reshape(NGRID, NGRID)

    # Score magnitude for heatmap
    score_mag = np.linalg.norm(res['score'].reshape(NGRID, NGRID, -1), axis=2)

    # Row 0: D_pull magnitude (denoiser target landscape)
    ax0 = axes[0, ei]
    im = ax0.pcolormesh(alpha_ax, beta_ax, score_mag.T,
                        cmap='magma', shading='auto')
    stride = max(1, NGRID // 12)
    ax0.quiver(A[::stride, ::stride], B[::stride, ::stride],
               disp_u[::stride, ::stride], disp_v[::stride, ::stride],
               color='white', alpha=0.7, scale=30, width=0.004)
    for mx, my, mlbl, mc in markers:
        ax0.plot(mx, my, 'o', color=mc, ms=7, mec='black', mew=0.8)
        ax0.text(mx, my + 0.2, mlbl, color=mc, fontsize=7, ha='center')
    ax0.set_xlim(alpha_ax[0], alpha_ax[-1])
    ax0.set_ylim(beta_ax[0],  beta_ax[-1])
    ax0.set_aspect('equal')
    ax0.set_title(f"ep {epoch:,}", fontsize=10)
    if ei == 0: ax0.set_ylabel('β (L2)', fontsize=9)
    ax0.spines['right'].set_visible(False)
    ax0.spines['top'].set_visible(False)

    # Row 1: D_pull projected onto v_ab (basin pull)
    Du_proj, _ = project_to_basis(res['D_pull'], v_ab, v_ac)
    Du_grid = Du_proj.reshape(NGRID, NGRID)
    vmax = np.percentile(np.abs(Du_grid), 95)
    ax1 = axes[1, ei]
    ax1.pcolormesh(alpha_ax, beta_ax, Du_grid.T,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
    ax1.contour(alpha_ax, beta_ax, Du_grid.T, levels=[0], colors='k', linewidths=0.8)
    for mx, my, mlbl, mc in markers:
        ax1.plot(mx, my, 'o', color=mc, ms=7, mec='black', mew=0.8)
    ax1.set_xlim(alpha_ax[0], alpha_ax[-1])
    ax1.set_ylim(beta_ax[0],  beta_ax[-1])
    ax1.set_aspect('equal')
    ax1.set_xlabel('α (L2)', fontsize=9)
    if ei == 0:
        ax1.set_ylabel('β (L2)', fontsize=9)
        ax1.text(-0.35, 0.5, r'$D_{pull}\cdot v_{ab}$', transform=ax1.transAxes,
                 fontsize=9, va='center', rotation=90)
    ax1.spines['right'].set_visible(False)
    ax1.spines['top'].set_visible(False)

axes[0, 0].text(-0.35, 0.5, 'Score\n|score|', transform=axes[0,0].transAxes,
                fontsize=9, va='center', rotation=90)
fig.suptitle(f"{EXP_NAME}  σ={VF_SIGMA}  three-sample plane", fontsize=12)
savefig_both(fig, '03_vector_field_checkpoints')
plt.show()

## Panel 4 — DSM Loss σ∈[0.2, 2.0] vs Training Step

In [ ]:
records = load_sigma_data(EXP_NAME)
steps = np.array([r['epoch'] for r in records])
sigma_grid = records[0]['sigma_grid']

# Only σ∈[0.2, 2.0] bin
smin, smax = 0.2, 2.0
fig, ax = plt.subplots(figsize=(9, 4))

SPLIT_STYLES_LOCAL = {
    'train':  dict(color='#2166ac', lw=1.8, label='Train (in-dist)'),
    'test':   dict(color='#d73027', lw=1.8, label='Test (valid, unseen)'),
    'random': dict(color='#555555', lw=1.4, ls='--', label='Random ±1'),
}
for split, style in SPLIT_STYLES_LOCAL.items():
    vals = np.array([bin_mean(r[f'loss_{split}'], sigma_grid, smin, smax) for r in records])
    mask = ~np.isnan(vals)
    ax.semilogy((steps + 1)[mask], vals[mask], **style)

ax.set_xscale('log')
ax.set_xlabel('Training step', fontsize=10)
ax.set_ylabel('MSE loss', fontsize=10)
ax.set_title(f"{EXP_NAME}  DSM loss  σ∈[0.2, 2.0]", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
fig.tight_layout()
savefig_both(fig, '04_dsm_loss_sigma02_2_vs_step')
plt.show()

## Panel 5 — DSM Loss vs σ at Key Checkpoints (log-log)

In [ ]:
# ep 49, 14251, 119377, 492388  (indices 10, 26, 32, 36)
fig, axes = plt.subplots(1, len(SIGMA_LOSS_IDX), figsize=(16, 4))
ep_labels = ['ep 49\n(pre rule)', 'ep 14,251\n(rule plateau)', 
             'ep 119,377\n(rule learned)', 'ep 492,388\n(mem onset)']
for ax, widx, lbl in zip(axes, SIGMA_LOSS_IDX, ep_labels):
    rec = records[widx]
    _draw_sigma_panel(ax, rec, lbl)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

fig.suptitle(f"{EXP_NAME}  DSM loss vs σ (log-log)", fontsize=12)
fig.tight_layout()
savefig_both(fig, '05_dsm_loss_vs_sigma_loglog')
plt.show()

## Panel 6 — Attractor Basin Profiles

In [ ]:
fig = basin_plot_profiles(
    cache_dir=BASIN_CACHE,
    epochs=BASIN_EPOCHS,
    epoch_labels=BASIN_LABELS,
    epoch_colors=BASIN_COLORS,
    sigma=BASIN_SIGMA,
    n_samples=BASIN_N,
    title=(f"{EXP_NAME}  Attractor basin profiles  σ={BASIN_SIGMA}  "
           f"N={BASIN_N} samples  (shading: 5–95% CI of mean)"),
)
savefig_both(fig, '06_attractor_basin_profiles')
plt.show()